# Week 2

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

df = pd.read_csv("../Dataset/MQTTEEB-D_cleaned_data.csv", low_memory=False)
print(df.isna().sum())

timestamp                   0
tcp_flags                   0
tcp_time_delta              0
tcp_len                     0
mqtt_conack_flags           0
mqtt_conflag_cleansess      0
mqtt_conflags               0
mqtt_dupflag                0
mqtt_hdrflags               0
mqtt_kalive                 0
mqtt_msg                    0
mqtt_qos                    0
label                     246
dtype: int64


Some rows have null value in "label" column.

So we will remove then, 246/222813 is not a big deal.

In [2]:
df = df.dropna()

The time could create a data leakage, so we remove them. 

We store it in the df_bis

In [3]:
df_bis = df[["timestamp", "tcp_time_delta"]]
df = df.drop(columns = ["timestamp", "tcp_time_delta"])

The model cannot work with hexadecimal, so we convert it in decimal.

Finally we convert the decimal number to strings; this won't be passed to the model, it's only for our own understanding. 

In [4]:
# mqtt_conack_flags --> Octet du variable header packet CONNACK: réponse du broker à un CONNACK 
# mqtt_conflags --> Octet du variable header packet CONNECT: réponse envoyé par un client lors d'un CONNECT
# mqtt_hdrflags --> Octet du Fixed header : CONNECT PUBLISH OU DISCONNECT

mqtt_hex_cols = ["mqtt_conack_flags", "mqtt_conflags", "mqtt_hdrflags", "tcp_flags"]

#for col in mqtt_hex_cols:
#    mqtt_hex_cols = ["mqtt_conack_flags", "mqtt_conflags", "mqtt_hdrflags", "tcp_flags"]

 #   for col in mqtt_hex_cols:
  #      df[f"{col}_int"] = df[col].apply(lambda x: int(str(x), 16))

for col in mqtt_hex_cols:
    df[f"{col}_int"] = [int(x, 16) for x in df[col]]

df["mqtt_msg_type"] = df["mqtt_hdrflags_int"] // 16

mqtt_packet_types = {
    1: "CONNECT", 2: "CONNACK", 3: "PUBLISH", 4: "PUBACK",
    8: "SUBSCRIBE", 9: "SUBACK", 12: "PINGREQ", 13: "PINGRESP", 14: "DISCONNECT"
}

df_bis["mqtt_msg_type_str"] = df["mqtt_msg_type"].map(mqtt_packet_types).fillna("OTHER")

tcp_flags_strings_list = []
#tcp_flags_int_list = []
for val in df["tcp_flags_int"]:
    #tcp_flags_int_list.append(val)
    tcp_flags_strings = ""

    if val >= 128: 
        tcp_flags_strings += "CWR "
        val -= 128
    if val >= 64: 
        tcp_flags_strings += "ECE "
        val -= 64
    if val >= 32: 
        tcp_flags_strings += "URG "
        val -= 32
    if val >= 16: 
        tcp_flags_strings += "ACK "
        val -= 16
    if val >= 8: 
        tcp_flags_strings += "PSH "
        val -= 8
    if val >= 4: 
        tcp_flags_strings += "RST "
        val -= 4
    if val >= 2: 
        tcp_flags_strings += "SYN "
        val -= 2
    if val == 1: 
        tcp_flags_strings += "FIN "
        val -= 1
    tcp_flags_strings_list.append(tcp_flags_strings)

conflags_strings_list = []
for val in df["mqtt_conflags_int"]:
    flags = ""
    if val >= 128:
        flags += "USER "
        val -= 128
    if val >= 64:
        flags += "PASS "
        val -= 64
    if val >= 32:
        flags += "WILL_RET "
        val -= 32
    if val >= 16:
        val -= 16  # Bit QoS haut
    if val >= 8:
        val -= 8   # Bit QoS bas
    if val >= 4:
        flags += "WILL_FLAG "
        val -= 4
    if val >= 2:
        flags += "CLEAN_SESS "
        val -= 2
    conflags_strings_list.append(flags.strip() if flags else "NONE")

df_bis["mqtt_conflags_strings"] = conflags_strings_list

#df["tcp_flags_int"] = tcp_flags_int_list
df_bis["tcp_flags_strings"] = tcp_flags_strings_list
df = df.drop(columns=[c for c in mqtt_hex_cols if c in df.columns])
    

In [5]:
df.head(10)

,tcp_len,mqtt_conflag_cleansess,mqtt_dupflag,mqtt_kalive,mqtt_msg,mqtt_qos,label,mqtt_conack_flags_int,mqtt_conflags_int,mqtt_hdrflags_int,tcp_flags_int,mqtt_msg_type
0,617,0,0,0,37372,2,legitimate,0,0,53,24,3
1,609,0,0,0,37373,2,legitimate,0,0,53,24,3
2,614,0,0,0,37374,2,legitimate,0,0,53,24,3
3,615,0,0,0,37375,2,legitimate,0,0,53,24,3
4,615,0,0,0,37376,2,legitimate,0,0,53,24,3
5,614,0,0,0,37377,2,legitimate,0,0,53,24,3
6,614,0,0,0,37378,2,legitimate,0,0,53,24,3
7,39,1,0,60,0,0,dos,0,130,16,24,1
8,39,1,0,60,0,0,dos,0,130,16,24,1
9,39,1,0,60,0,0,dos,0,130,16,24,1


In [6]:
df_bis.head(10)

,timestamp,tcp_time_delta,mqtt_msg_type_str,mqtt_conflags_strings,tcp_flags_strings
0,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
1,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
2,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
3,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
4,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
5,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
6,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
7,1.725852e+09,1.725852e+09,CONNECT,USER CLEAN_SESS,ACK PSH
8,1.725852e+09,1.725852e+09,CONNECT,USER CLEAN_SESS,ACK PSH
9,1.725852e+09,1.725852e+09,CONNECT,USER CLEAN_SESS,ACK PSH


Here we don't want to predict which type of attack we have detect, we just want to detect if the row is an attack or legitimate. 

In [7]:
target = []
for l in df["label"]:
    if l != "legitimate":
        target.append(1)
    else: 
        target.append(0)

df["target"] = target
df_bis["label"] = df["label"]
df = df.drop(columns = ["label"])

In [8]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, stratify=y, random_state=42)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, f1_score, confusion_matrix
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)

scaler = StandardScaler()
X_train_stand = scaler.fit_transform(X_train)
X_test_stand = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_stand, y_train)

y_pred = model.predict(X_test_stand)

print(confusion_matrix(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test,y_pred))
print("Balanced accuracy: ", balanced_accuracy_score(y_test,y_pred))
print("F1 - score: ", f1_score(y_test,y_pred))
print("Precision", precision_score (y_test , y_pred))
print("Recall", recall_score (y_test , y_pred))


[[ 1794  2595]
 [ 4442 46811]]
Accuracy:  0.8735307860968333
Balanced accuracy:  0.661040524057015
F1 - score:  0.9300907022720274
Precision 0.9474760150588998
Recall 0.9133319025227792


The best metric here is the balanced accuracy, because we are in a imbalanced dataset. 

In [11]:
print(df.isna().sum())
print(df_bis.isna().sum())

tcp_len                   0
mqtt_conflag_cleansess    0
mqtt_dupflag              0
mqtt_kalive               0
mqtt_msg                  0
mqtt_qos                  0
mqtt_conack_flags_int     0
mqtt_conflags_int         0
mqtt_hdrflags_int         0
tcp_flags_int             0
mqtt_msg_type             0
target                    0
dtype: int64
timestamp                0
tcp_time_delta           0
mqtt_msg_type_str        0
mqtt_conflags_strings    0
tcp_flags_strings        0
label                    0
dtype: int64
